In [7]:
import casadi as ca
import numpy as np

# -----------------------------------------------------------
# Generalized coordinates and rates
# -----------------------------------------------------------
q  = ca.SX.sym('q', 8)   # [x, y, z, psi, th, phi, a, b]
qd = ca.SX.sym('qd', 8)
x, y, z, psi, th, phi, a, b = q[0], q[1], q[2], q[3], q[4], q[5], q[6], q[7]
xd, yd, zd, psid, thd, phid, ad, bd = qd[0], qd[1], qd[2], qd[3], qd[4], qd[5], qd[6], qd[7]

# -----------------------------------------------------------
# Parameters
# -----------------------------------------------------------
rx, ry, rz = ca.SX.sym('rx'), ca.SX.sym('ry'), ca.SX.sym('rz')
m_h, m_r, m, g = ca.SX.sym('m_h'), ca.SX.sym('m_r'), ca.SX.sym('m'), ca.SX.sym('g')

I_hxx, I_hyy, I_hzz = ca.SX.sym('I_hxx'), ca.SX.sym('I_hyy'), ca.SX.sym('I_hzz')
I_rxx, I_ryy, I_rzz = ca.SX.sym('I_rxx'), ca.SX.sym('I_ryy'), ca.SX.sym('I_rzz')
I_h = ca.diag(ca.vertcat(I_hxx, I_hyy, I_hzz))
I_r = ca.diag(ca.vertcat(I_rxx, I_ryy, I_rzz))

# -----------------------------------------------------------
# Rotation matrices
# -----------------------------------------------------------
def Rx(a): return ca.vertcat(
    ca.horzcat(1, 0, 0),
    ca.horzcat(0, ca.cos(a), -ca.sin(a)),
    ca.horzcat(0, ca.sin(a),  ca.cos(a))
)
def Ry(a): return ca.vertcat(
    ca.horzcat(ca.cos(a), 0, ca.sin(a)),
    ca.horzcat(0, 1, 0),
    ca.horzcat(-ca.sin(a), 0, ca.cos(a))
)
def Rz(a): return ca.vertcat(
    ca.horzcat(ca.cos(a), -ca.sin(a), 0),
    ca.horzcat(ca.sin(a),  ca.cos(a), 0),
    ca.horzcat(0, 0, 1)
)

R_SH = Rz(psi) @ Ry(th) @ Rx(phi)
R_Hr = Rx(a)
R_rp = Rx(b)
R_Sp = R_SH @ R_Hr @ R_rp

# -----------------------------------------------------------
# Positions
# -----------------------------------------------------------
rH_S = ca.vertcat(x, y, z)
rP_p = ca.vertcat(rx, ry, rz)
rP_S = rH_S + R_Sp @ rP_p

# Velocities
J_rH = ca.jacobian(rH_S, q)
J_rP = ca.jacobian(rP_S, q)
rH_S_dot = J_rH @ qd
rP_S_dot = J_rP @ qd

# -----------------------------------------------------------
# Angular velocities (analytic ZYX mapping)
# -----------------------------------------------------------
J_zyx = ca.vertcat(
    ca.horzcat(1, 0, -ca.sin(th)),
    ca.horzcat(0, ca.cos(phi), ca.sin(phi)*ca.cos(th)),
    ca.horzcat(0, -ca.sin(phi), ca.cos(phi)*ca.cos(th))
)
omega_H_H = J_zyx @ ca.vertcat(phid, thd, psid)
omega_r_r = ca.vertcat(ad, 0, 0)

# -----------------------------------------------------------
# Kinetic & potential energy
# -----------------------------------------------------------
T = 0.5*m_h*(rH_S_dot.T @ rH_S_dot) \
  + 0.5*m_r*(rP_S_dot.T @ rP_S_dot) \
  + 0.5*(omega_H_H.T @ I_h @ omega_H_H) \
  + 0.5*(omega_r_r.T @ I_r @ omega_r_r)

V = m * g * rP_S[2]
L = T - V

# -----------------------------------------------------------
# Euler–Lagrange equations
# -----------------------------------------------------------
dLdq   = ca.jacobian(L, q).T
dLdqd  = ca.jacobian(L, qd).T
d_dt_dLdqd = ca.jacobian(dLdqd, q) @ qd  # since L has no explicit t
EL = ca.simplify(d_dt_dLdqd - dLdq)

params = ca.vertcat(rx, ry, rz, m_h, m_r, m, g,
                    I_hxx, I_hyy, I_hzz, I_rxx, I_ryy, I_rzz)

EL_func = ca.Function('EL_func', [q, qd, params], [EL])

# -----------------------------------------------------------
# Test numeric evaluation
# -----------------------------------------------------------
q0 = np.zeros(8)
qd0 = np.zeros(8)
params_val = np.array([0, 0, -1, 1, 1, 1, 9.81, 1, 1, 1, 1, 1, 1])

print("Euler–Lagrange equations (evaluated at q=0, qd=0):")
print(np.array(EL_func(q0, qd0, params_val)))


Euler–Lagrange equations (evaluated at q=0, qd=0):
[[0.  ]
 [0.  ]
 [9.81]
 [0.  ]
 [0.  ]
 [0.  ]
 [0.  ]
 [0.  ]]
